# Simple: Utility Metrics with PAMOLA.CORE

**Goal**: Learn utility metrics basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply classification and regression utility metrics using execute()
- Compare original vs. transformed model performance
- Understand privacy-utility tradeoffs

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── metrics/
        └── 01_utility_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.metrics.operations.utility_ops import UtilityMetricOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load original and transformed datasets from `examples/data_examples/`
- Auto-create sample data if files don't exist
- Inspect both dataset structures

**Expected output:** Two DataFrames (original and transformed) with classification/regression features

In [ ]:
examples_dir = project_root / 'examples'
original_data_path = examples_dir / 'data_examples' / 'sample.csv'
transformed_data_path = examples_dir / 'data_examples' / 'sample.csv'

if not original_data_path.exists():
    print("⚠️  Original file not found, creating sample data...")
    original_data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample original data with classification and regression targets
    np.random.seed(42)
    n_records = 100
    
    original_sample = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'age': np.random.randint(18, 80, n_records),
        'income': np.random.normal(50000, 15000, n_records).astype(int),
        'education_years': np.random.randint(8, 20, n_records),
        'hours_worked': np.random.randint(20, 60, n_records),
        'classification_target': np.random.choice([0, 1, 2], n_records),  # 3 classes
        'regression_target': np.random.normal(100, 20, n_records)
    })
    original_sample.to_csv(original_data_path, index=False)
    print(f"✓ Original sample data created")

if not transformed_data_path.exists():
    print("⚠️  Transformed file not found, creating sample data...")
    
    # Create transformed data with privacy noise
    np.random.seed(43)
    original_df = pd.read_csv(original_data_path)
    
    transformed_sample = original_df.copy()
    # Add noise to simulate privacy transformation
    transformed_sample['age'] = transformed_sample['age'] + np.random.randint(-3, 4, len(transformed_sample))
    transformed_sample['income'] = transformed_sample['income'] + np.random.normal(0, 2000, len(transformed_sample)).astype(int)
    transformed_sample['education_years'] = transformed_sample['education_years'] + np.random.randint(-1, 2, len(transformed_sample))
    transformed_sample['hours_worked'] = transformed_sample['hours_worked'] + np.random.randint(-3, 4, len(transformed_sample))
    transformed_sample['regression_target'] = transformed_sample['regression_target'] + np.random.normal(0, 5, len(transformed_sample))
    
    transformed_sample.to_csv(transformed_data_path, index=False)
    print(f"✓ Transformed sample data created")

original_df = read_csv(original_data_path)
transformed_df = read_csv(transformed_data_path)

print(f"\n📊 Original Dataset: {len(original_df)} records, {len(original_df.columns)} columns")
print(f"🔍 First 5 records:")
print(original_df.head())

print(f"\n📊 Transformed Dataset: {len(transformed_df)} records, {len(transformed_df.columns)} columns")
print(f"🔍 First 5 records:")
print(transformed_df.head())

print(f"\n📈 Column Statistics (Original):")
for col in original_df.columns:
    dtype_str = str(original_df[col].dtype)
    if pd.api.types.is_numeric_dtype(original_df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): min={original_df[col].min():.2f}, max={original_df[col].max():.2f}, mean={original_df[col].mean():.2f}")
    else:
        print(f"  {col:25s} ({dtype_str:10s}): {original_df[col].nunique()} unique values")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with both DataFrames
- Configure progress tracker
- **Configure dataset names** for processing

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset",
        "utility_metrics": ["classification", "regression"],  # classification, regression
        "columns": ["years_of_experience", "current_salary_cad"],
        "metric_params": {
            "regression": {
                "models": [
                    "logistic"
                ],
                "metrics": [
                    "r2",
                    "mae"
                ],
                "cv_folds": 1,
                "key_fields": [
                    "years_of_experience"
                ],
                "aggregation": "sum",
                "value_field": "current_salary_cad"
                },
            "classification": {
                "models": [
                    "logistic"
                ],
                "metrics": [
                    "accuracy",
                    "f1"
                ],
                "cv_folds": 1,
                "value_field": "current_salary_cad"
            }
        }
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset",
        "utility_metrics": ["classification", "regression"],  # classification, regression
        "columns": ["years_of_experience", "resume_id"],
        "metric_params": {
            "regression": {
                "models": [
                    "linear"
                ],
                "metrics": [
                    "r2",
                    "mae"
                ],
                "cv_folds": 1,
                "key_fields": [
                    "resume_id"
                ],
                "aggregation": "sum",
                "value_field": "years_of_experience"
                },
            "classification": {
                "models": [
                    "logistic"
                ],
                "metrics": [
                    "accuracy",
                    "f1"
                ],
                "cv_folds": 1,
                "value_field": "years_of_experience"
            }
        }
    }

kwargs = create_config_kwargs()
utility_metrics = kwargs["utility_metrics"]
metric_params = kwargs["metric_params"]
columns = kwargs["columns"]
original_dataset_name = kwargs["original_dataset_name"]
transformed_dataset_name = kwargs["transformed_dataset_name"]

print(f"\n🔍 Validating dataset configuration...\n")
print(f"✓ Original dataset name: '{original_dataset_name}'")
print(f"  Records: {len(original_df)}")
print(f"  Columns: {list(original_df.columns)}")
print(f"✓ Utility metrics: {', '.join(utility_metrics)}")
print(f"✓ Columns to compare: {', '.join(columns)}")
print(f"✓ Metric params: {', '.join(metric_params)}")

print(f"\n✓ Transformed dataset name: '{transformed_dataset_name}'")
print(f"  Records: {len(transformed_df)}")
print(f"  Columns: {list(transformed_df.columns)}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="utility_simple_001",
    task_type="utility_metrics",
    description="Utility metrics calculation for classification and regression",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {
    "original_dataset_name": original_dataset_name,
    "transformed_dataset_name": transformed_dataset_name
}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource with both datasets
data_source = DataSource(dataframes={
    "original_dataset": original_df,
    "transformed_dataset": transformed_df
})
print("✓ DataSource created with both datasets")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description="Processing utility metrics",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create UtilityMetricOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `utility_metrics=['classification', 'regression']`: Metrics to calculate
- `metric_params`: Configuration for each metric type
  - Classification: models (logistic, rf, svm), metrics (accuracy, f1, roc_auc)
  - Regression: models (linear, rf, svr), metrics (r2, mae, mse, rmse)
- `columns`: Features to use for model training
- `column_mapping`: Map columns between datasets (if needed)
- `cv_folds=5`: Cross-validation folds
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = UtilityMetricOperation(
    # Core parameters
    name='utility_metrics',
    utility_metrics=utility_metrics,
    
    # Metric-specific parameters
    metric_params=metric_params,
    
    # Feature columns (exclude record_id and target fields)
    columns=columns,
    
    # Column mapping (if column names differ between datasets)
    column_mapping={},
    
    # Metric options
    normalize=True,
    confidence_level=0.95,
    sample_size=None,  # Use full dataset
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Name:                 {operation.name}")
print(f"  Utility metrics:      {operation.utility_metrics}")
print(f"  Classification models: {operation.metric_params['classification']['models']}")
print(f"  Regression models:    {operation.metric_params['regression']['models']}")
print(f"  Feature columns:      {len(operation.columns)} columns")
print(f"  Visualizations:       {operation.generate_visualization}")
print(f"  Save output:          {operation.save_output}")
print(f"  Force recalc:         {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing Utility Metrics Operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Review metrics from operation result
- Compare classification and regression performance
- Understand privacy-utility tradeoffs

**Generated artifacts:**
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Display result metrics
if result.metrics:
    print("\n" + "=" * 80)
    print("📊 UTILITY METRICS SUMMARY")
    print("=" * 80)
    
    # Basic metrics
    print("\n🔍 Processing Metrics:")
    for key in ['duration_seconds', 'records_processed', 'records_per_second']:
        if key in result.metrics:
            print(f"  {key:30s}: {result.metrics[key]}")
    
    print("\n" + "=" * 80)
    print("✨ INSIGHTS")
    print("=" * 80)
    print("")
    print("💡 Utility metrics quantify how much the transformed data retains")
    print("   the statistical properties of the original data.")
    print("")
    print("📈 Higher scores indicate better utility preservation:")
    print("   • Classification: accuracy, F1, ROC-AUC closer to original")
    print("   • Regression: R², MAE, MSE, RMSE closer to original")
    print("")
    print("⚖️  Privacy-Utility Tradeoff:")
    print("   • More privacy protection → potentially lower utility")
    print("   • Less privacy protection → higher utility")
    print("   • Goal: Find optimal balance for your use case")
else:
    print("⚠️  No metrics available in result")

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Classification and regression performance metrics
2. **Visualizations**: Model performance charts and precision-recall curves

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / "metrics"

if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")

else:
    metrics_files = list(metrics_dir.glob("*.json"))

    if not metrics_files:
        print("⚠️  No metrics files found")

    else:
        # 1) Exclude data_types*
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"\n📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB")
        print("=" * 80)

        try:
            with open(latest_metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # ─────────────────────────────────────────────
            # METADATA
            # ─────────────────────────────────────────────
            print("\n📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if "operation" in metadata:
                op = metadata["operation"]
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # ─────────────────────────────────────────────
            # UTILITY METRICS
            # ─────────────────────────────────────────────
            print("\n📊 UTILITY METRICS:")
            print("=" * 80)

            # ───── Classification ─────
            if "classification" in metrics:
                print("\n🎯 CLASSIFICATION METRICS:")
                print("-" * 60)

                for model, model_metrics in metrics["classification"].items():
                    print(f"\n  Model: {model.upper()}")
                    for k, v in model_metrics.items():
                        if isinstance(v, (int, float)):
                            print(f"    {k:25s}: {v:.6f}")
                        else:
                            print(f"    {k:25s}: {v}")

            else:
                print("\n🎯 CLASSIFICATION METRICS: N/A")

            # ───── Regression ─────
            if "regression" in metrics:
                print("\n📈 REGRESSION METRICS:")
                print("-" * 60)

                for model, model_metrics in metrics["regression"].items():
                    print(f"\n  Model: {model.upper()}")
                    for k, v in model_metrics.items():
                        if isinstance(v, (int, float)):
                            print(f"    {k:25s}: {v:.6f}")
                        else:
                            print(f"    {k:25s}: {v}")

            else:
                print("\n📈 REGRESSION METRICS: N/A")

            # ─────────────────────────────────────────────
            # PERFORMANCE
            # ─────────────────────────────────────────────
            print("\n⚡ PERFORMANCE:")
            print("-" * 60)

            print(f"   Duration (s):        {metrics.get('duration_seconds', 0):.4f}")
            print(f"   Records Processed:  {metrics.get('records_processed', 0):,}")
            print(f"   Records / Second:   {metrics.get('records_per_second', 0):.2f}")

            if "sample_size" in metrics:
                if metrics["sample_size"] is None:
                    print("   Sample Size:        N/A")
                else:
                    print(f"   Sample Size:        {metrics['sample_size']:,}")

            if "total_original_records" in metrics:
                print(f"   Original Records:   {metrics['total_original_records']:,}")
            if "total_transformed_records" in metrics:
                print(f"   Transformed Records:{metrics['total_transformed_records']:,}")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. VISUALIZATIONS (NEWEST BATCH)
print("\n\n2️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ Load original and transformed data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure utility metrics with classification and regression models  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- **Utility metrics** measure how well transformed data preserves statistical properties of the original
- **Classification metrics** (accuracy, F1, ROC-AUC) evaluate categorical prediction quality
- **Regression metrics** (R², MAE, MSE, RMSE) evaluate numerical prediction quality
- **Privacy-utility tradeoff**: More privacy protection typically reduces utility, requiring careful balance
- **Cross-validation** provides robust, reliable performance estimates across data splits
- **Model comparison**: Multiple models (logistic, RF, SVM/SVR) provide comprehensive evaluation

**Understanding the Results:**
- Scores closer to 1.0 (or closer to original) indicate better utility preservation
- Lower error metrics (MAE, MSE, RMSE) indicate better regression performance
- Precision-recall curves show classification performance across different thresholds

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_utility_metrics_advanced.ipynb`** includes:
- Custom model configurations with hyperparameter tuning
- Advanced cross-validation strategies (nested CV, stratified K-fold)
- Multi-model ensemble analysis and comparison
- Feature importance and selection techniques
- Large-scale processing with Dask parallelization
- Integration with privacy profiling results

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 📊 [Utility Metrics Guide](../../docs/utility_metrics.md)